In [49]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter
import string

In [50]:
nltk.download("punkt")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\guill\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [51]:
# Load data
df = pd.read_excel("../data/noticiasQA2.xlsx")

In [52]:
# Load Spanish QA model
qa_model = pipeline(
    "question-answering",
    model= "mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es",
    tokenizer= "mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es"
)

Some weights of the model checkpoint at mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [53]:
# Simple text normalization
def normalize(text):
    text = text.lower().translate(str.maketrans("", "", string.punctuation))
    return word_tokenize(text)

In [54]:
# Compute F1 score
def f1_score(prediction, ground_truth):
    pred_tokens = normalize(prediction)
    truth_tokens = normalize(ground_truth)

    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())

    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens)
    if num_same == 0:
        return 0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(truth_tokens)
    return 2 * (precision * recall) / (precision + recall)

In [55]:
# Compute Exact Match
def exact_match(prediction, ground_truth):
    return int(normalize(prediction) == normalize(ground_truth))

# Metrics
total = len(df)
em_total = 0
f1_total = 0

In [56]:
# Evaluate
for _, row in tqdm(df.iterrows(), total=total):
    context = row["context"]
    question = row["question"]
    ground_truth = str(row["answer"])

    try:
        prediction = qa_model({
            "context": context,
            "question": question
        })["answer"]

        em_total += exact_match(prediction, ground_truth)
        f1_total += f1_score(prediction, ground_truth)

    except Exception as e:
        print("Error:", e)

# Final results
print(f"\nExact Match: {(em_total / total) * 100:.2f}%")
print(f"F1 Score: {(f1_total / total) * 100:.2f}%")


  0%|          | 0/21 [00:00<?, ?it/s]C:\Users\guill\Documents\GitHub\nlp-media-framing\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(
100%|██████████| 21/21 [00:09<00:00,  2.12it/s]


Exact Match: 33.33%
F1 Score: 76.48%
